In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# Install librosa for audio processing
!pip install -q librosa

import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm 

# 1. Define Paths (Standard Kaggle Input Paths)
# Note: Kaggle datasets are read-only, so we read from /input and write to /working
BASE_INPUT_DIR = '/kaggle/input/asvpoof-2019-dataset/LA/LA'
TRAIN_AUDIO_DIR = f'{BASE_INPUT_DIR}/ASVspoof2019_LA_train/flac'
PROTOCOL_FILE = f'{BASE_INPUT_DIR}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'

# 2. Create Output Directories
# We need separate folders for "bonafide" (Real) and "spoof" (Fake) images
OUTPUT_DIR = '/kaggle/working/spectrograms'
os.makedirs(f'{OUTPUT_DIR}/bonafide', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/spoof', exist_ok=True)

print("Setup Complete. Directories created.")

Setup Complete. Directories created.


In [5]:
# 1. Read the Label File
# Columns: Speaker_ID, Audio_FileName, System_ID, null, Key (bonafide/spoof)
labels = pd.read_csv(PROTOCOL_FILE, sep=" ", header=None, names=['spk_id', 'fname', 'sys_id', 'null', 'key'])

# 2. Balance the Dataset (Select 1500 of each to keep training fast)
bonafide_files = labels[labels['key'] == 'bonafide']['fname'].values[:1500]
spoof_files = labels[labels['key'] == 'spoof']['fname'].values[:1500]

print(f"Selected {len(bonafide_files)} Real tracks and {len(spoof_files)} Fake tracks.")

# 3. Define the Spectrogram Function
def create_spectrogram(filename, label):
    audio_path = f"{TRAIN_AUDIO_DIR}/{filename}.flac"
    save_path = f"{OUTPUT_DIR}/{label}/{filename}.png"
    
    # Skip if already exists
    if os.path.exists(save_path): return

    try:
        # Load Audio (Limit to 4 seconds)
        y, sr = librosa.load(audio_path, sr=16000, duration=4.0)
        
        # Pad audio if it's shorter than 4 seconds
        target_length = 16000 * 4
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), 'constant')

        # Generate Mel Spectrogram
        # fmax=8000 captures the high-freq artifacts where Deepfakes fail
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Save as tiny Image (128x128 is plenty for MobileNet)
        plt.figure(figsize=(2, 2))
        librosa.display.specshow(mel_spec_db, sr=sr, fmax=8000)
        plt.axis('off')
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
        plt.close()
    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 4. Run the Processing Loop
print("Generating Spectrograms... (This will take about 5-8 minutes)")

for f in tqdm(bonafide_files, desc="Processing Real"):
    create_spectrogram(f, "bonafide")

for f in tqdm(spoof_files, desc="Processing Fake"):
    create_spectrogram(f, "spoof")

print("Spectrogram generation complete!")

Selected 1500 Real tracks and 1500 Fake tracks.
Generating Spectrograms... (This will take about 5-8 minutes)


Processing Fake: 100%|██████████| 1500/1500 [00:00<00:00, 141155.82it/s]

Spectrogram generation complete!


In [6]:
# 1. Setup Data Loaders
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

print("Loading images...")
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    OUTPUT_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary' # 0 = bonafide, 1 = spoof (Alphabetical)
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    OUTPUT_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# 2. Build the VoxSentinel Model
base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base for Transfer Learning

inputs = Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x) # Regularization
outputs = Dense(1, activation='sigmoid')(x) # Output: 0-1 Probability

model = Model(inputs, outputs)

# 3. Compile
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 4. Train
print("Starting Training...")
history = model.fit(train_ds, validation_data=val_ds, epochs=8)
print("Training Finished!")

Loading images...
Found 3000 files belonging to 2 classes.
Using 2400 files for training.
Found 3000 files belonging to 2 classes.
Using 600 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Starting Training...
Epoch 1/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 29s 316ms/step - accuracy: 0.7183 - loss: 0.5589 - val_accuracy: 0.9817 - val_loss: 0.1262
Epoch 2/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 289ms/step - accuracy: 0.9730 - loss: 0.1199 - val_accuracy: 0.9867 - val_loss: 0.0790
Epoch 3/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 287ms/step - accuracy: 0.9812 - loss: 0.0790 - val_accuracy: 0.9900 - val_loss: 0.0589
Epoch 4/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 293ms/step - accuracy: 0.9830 - loss: 0.0627 - val_accuracy: 0.9867 - val_loss: 0.0511
Epoch 5/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 23s 308ms/step - accuracy: 0.9901 - loss: 0.0490 - val_accuracy: 0.9917 - val_loss: 0.0415
Epoch 6/8
75/75 ━━━━━━━━━━━━━━━━━━━━ 23s 309ms/step - accuracy: 0.9900 - loss: 0.0479 - val_accuracy: 0.9900 - val_loss: 0.0353

In [7]:
# 1. Convert to TFLite (Optimized for CPU/Mobile)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 2. Save the file
with open('voxsentinel.tflite', 'wb') as f:
    f.write(tflite_model)

print("SUCCESS! Model saved as 'voxsentinel.tflite'")

INFO:tensorflow:Assets written to: /tmp/tmps3gn7vk8/assets


INFO:tensorflow:Assets written to: /tmp/tmps3gn7vk8/assets


Saved artifact at '/tmp/tmps3gn7vk8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_308')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137566464658896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566464671376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566464669648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566464667536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566464655824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566479206160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566479213264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566479207120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566479207504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137566479219600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1375664792

W0000 00:00:1771065996.103687      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1771065996.103764      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1771065996.306062      55 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


In [8]:
from IPython.display import FileLink
FileLink(r'voxsentinel.tflite')

/kaggle/working/voxsentinel.tflite